# Model Experiments — Sensor Fault Detection

Comparing classifiers on the UCI SECOM dataset

In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

df = pd.read_csv("wafer_secom_cleaned.csv")
X = df.drop(columns=["Good/Bad"])
y = np.where(df["Good/Bad"] == -1, 0, 1)  # 0 = Good, 1 = Faulty

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []  # every model's summary row goes here

In [20]:
def evaluate_model(name, model, X=X, y=y, cv=cv):
    """Runs 5-fold stratified CV, prints a report, stores the summary in `results`."""
    recalls, precisions, f1s = [], [], []
    tn = fp = fn = tp = 0

    for train_idx, test_idx in cv.split(X, y):
        model.fit(X.iloc[train_idx], y[train_idx])
        preds = model.predict(X.iloc[test_idx])
        y_true = y[test_idx]

        recalls.append(recall_score(y_true, preds, zero_division=0))
        precisions.append(precision_score(y_true, preds, zero_division=0))
        f1s.append(f1_score(y_true, preds, zero_division=0))
        cm = confusion_matrix(y_true, preds, labels=[0, 1])
        tn += cm[0, 0]; fp += cm[0, 1]; fn += cm[1, 0]; tp += cm[1, 1]

    print(f"{name}")
    print(f"  Recall:    {np.mean(recalls):.3f} ± {np.std(recalls):.3f}")
    print(f"  Precision: {np.mean(precisions):.3f} ± {np.std(precisions):.3f}")
    print(f"  F1:        {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    print(f"  Faults caught: {tp}/{tp+fn}   False alarms: {fp}/{fp+tn}")

    results.append({
        "model": name, "recall": np.mean(recalls), "recall_std": np.std(recalls),
        "precision": np.mean(precisions), "f1": np.mean(f1s),
        "faults_caught": f"{tp}/{tp+fn}", "false_alarms": f"{fp}/{fp+tn}",
    })

In [21]:
from xgboost import XGBClassifier

evaluate_model("XGBoost (unweighted)", XGBClassifier(eval_metric="logloss", random_state=42))

XGBoost (unweighted)
  Recall:    0.010 ± 0.019
  Precision: 0.100 ± 0.200
  F1:        0.017 ± 0.035
  Faults caught: 1/104   False alarms: 2/1463


In [22]:
# XGBoost with scale_pos_weight
evaluate_model(
    "XGBoost (scale_pos_weight)",
    XGBClassifier(eval_metric="logloss", scale_pos_weight=(y==0).sum()/(y==1).sum(), random_state=42)
)

XGBoost (scale_pos_weight)
  Recall:    0.019 ± 0.023
  Precision: 0.140 ± 0.196
  F1:        0.033 ± 0.040
  Faults caught: 2/104   False alarms: 8/1463


In [23]:
# SVC, both variants
from sklearn.svm import SVC

evaluate_model("SVC (unweighted)", SVC(random_state=42))
evaluate_model("SVC (class_weight=balanced)", SVC(class_weight="balanced", random_state=42))

SVC (unweighted)
  Recall:    0.000 ± 0.000
  Precision: 0.000 ± 0.000
  F1:        0.000 ± 0.000
  Faults caught: 0/104   False alarms: 0/1463
SVC (class_weight=balanced)
  Recall:    0.403 ± 0.113
  Precision: 0.065 ± 0.012
  F1:        0.111 ± 0.021
  Faults caught: 42/104   False alarms: 604/1463


In [24]:
# BalancedRandomForest
from imblearn.ensemble import BalancedRandomForestClassifier

evaluate_model("BalancedRandomForest", BalancedRandomForestClassifier(random_state=42))

BalancedRandomForest
  Recall:    0.230 ± 0.062
  Precision: 0.189 ± 0.048
  F1:        0.207 ± 0.054
  Faults caught: 24/104   False alarms: 103/1463


In [25]:
# RUSBoost
from imblearn.ensemble import RUSBoostClassifier

evaluate_model("RUSBoost (max_depth=4)", RUSBoostClassifier(
    n_estimators=50,
    estimator=__import__("sklearn.tree", fromlist=["DecisionTreeClassifier"]).DecisionTreeClassifier(max_depth=4),
    random_state=42
))

RUSBoost (max_depth=4)
  Recall:    0.376 ± 0.115
  Precision: 0.157 ± 0.046
  F1:        0.210 ± 0.039
  Faults caught: 39/104   False alarms: 247/1463


In [26]:
# EasyEnsembleClassifier
from imblearn.ensemble import EasyEnsembleClassifier

evaluate_model("EasyEnsembleClassifier", EasyEnsembleClassifier(random_state=42))


EasyEnsembleClassifier
  Recall:    0.683 ± 0.093
  Precision: 0.129 ± 0.009
  F1:        0.217 ± 0.016
  Faults caught: 71/104   False alarms: 478/1463


In [27]:
results_df = pd.DataFrame(results).sort_values("recall", ascending=False)
results_df

,model,recall,recall_std,precision,f1,faults_caught,false_alarms
6,EasyEnsembleClassifier,0.682857,0.092611,0.128976,0.216708,71/104,478/1463
3,SVC (class_weight=balanced),0.403333,0.113325,0.064573,0.111031,42/104,604/1463
5,RUSBoost (max_depth=4),0.376190,0.115077,0.156848,0.209802,39/104,247/1463
4,BalancedRandomForest,0.230476,0.062131,0.188707,0.206910,24/104,103/1463
1,XGBoost (scale_pos_weight),0.019048,0.023328,0.140000,0.032776,2/104,8/1463
0,XGBoost (unweighted),0.009524,0.019048,0.100000,0.017391,1/104,2/1463
2,SVC (unweighted),0.000000,0.000000,0.000000,0.000000,0/104,0/1463


## Conclusion

`EasyEnsembleClassifier` is the clear choice: highest recall (68.3%) by a wide margin, catching 71 of 104 real faults in cross-validation, while remaining reasonably stable across folds (recall std 0.093).

The trade-off is a real one — 478 of 1,463 good wafers get flagged as false alarms (32.7%). This is accepted here on the assumption that this model feeds a human review step rather than making unreviewed automatic accept/reject decisions: in semiconductor QA, a missed fault (shipped defect) is assumed to cost more than an extra manual inspection.

Every other imbalance-handling approach tried (SVC balanced, RUSBoost, BalancedRandomForest, XGBoost with scale_pos_weight) caught meaningfully fewer real faults.